# 🎯 02 - Center and Spread

Module 06 used `.describe()` as a first look, to spot problems. Now we read it properly. Every numeric column can be summarized with two questions:

1. **Where is the middle?** (center: mean, median, mode)
2. **How spread out are the values?** (spread: range, standard deviation, IQR, percentiles)

Two columns can have the same center and completely different spreads, and a single number is never the whole story.

## ✅ Learning Goals
- Compute the mean, median, and mode with pandas methods, and explain when they disagree
- Show that the median is **robust** to outliers and the mean is not
- Compute range, variance, standard deviation, IQR, and any percentile
- Explain `ddof`, and why pandas and NumPy give different standard deviations by default
- Read every row of `.describe()`, including what `count` tells you about missing data
- See how filling missing values (Module 06) changes the statistics

In [1]:
import numpy as np
import pandas as pd

titanic = pd.read_csv('titanic.csv')
fare = titanic['Fare']
age = titanic['Age']

## 1. Center: Mean, Median, Mode
Every pandas `Series` has these built in, so no NumPy or SciPy is needed:

| Statistic | Method | Meaning |
|---|---|---|
| Mean | `.mean()` | The sum divided by the count. The "balance point." |
| Median | `.median()` | The middle value after sorting. Half the values are below it. |
| Mode | `.mode()` | The most common value (or values). |

In [2]:
print(f'Fare mean:   {fare.mean():.2f}')
print(f'Fare median: {fare.median():.2f}')
print('Fare mode:  ', fare.mode().tolist())

Fare mean:   32.20
Fare median: 14.45
Fare mode:   [8.05]


The mean fare is more than **twice** the median. That gap is a signal, and the next section shows what causes it.

`.mode()` returns a **Series**, not a single number, because a column can have a tie for most common value. Use `.mode()[0]` when you need just one.

## 2. Robustness: Who Gets Pulled by Outliers?
A statistic is **robust** if a few extreme values barely move it. Watch what happens to a small, made-up set of fares when one very expensive ticket is added:

In [3]:
fares = pd.Series([8, 10, 12, 15, 20])
with_outlier = pd.Series([8, 10, 12, 15, 20, 500])

print('Without outlier: mean', fares.mean(), '| median', fares.median())
print('With outlier:    mean', round(with_outlier.mean(), 1), '| median', with_outlier.median())

Without outlier: mean 13.0 | median 12.0
With outlier:    mean 94.2 | median 13.5


One value moved the mean from 13 to about 94. The median barely changed (12 → 13.5), because it only cares about the **middle position**, not how far away the extreme values are.

That's what is happening in the real Titanic fares: a small number of very expensive 1st-class tickets pull the mean far above what a typical passenger paid.

| Use the… | When… |
|---|---|
| **Median** | the data are lopsided or have extreme values (incomes, prices, house sizes) |
| **Mean** | the data are roughly symmetric, or you need a total (mean × count = total) |
| **Mode** | the data are categories, or a few repeated values (most common shoe size) |

✅ **Your Turn:** Compute the mean and median of `age`. Are they close? What does that suggest about whether age has extreme values?

In [4]:
# Your turn here
print(age.mean())
print(age.median())

# both mean and median give about the same value, so the data is roughly symmetric

29.69911764705882
28.0


## 3. Spread: Range, Variance, Standard Deviation
- **Range** = max − min. Easy, but it depends on only the two most extreme values.
- **Variance** = the average squared distance from the mean. Its units are squared (pounds²), which is hard to interpret.
- **Standard deviation (std)** = the square root of the variance, back in the original units. Roughly: how far a typical value is from the mean.

In [5]:
print(f'Age range: {age.max() - age.min():.2f} years ({age.min()} to {age.max()})')
print(f'Age variance: {age.var():.2f} (years squared)')
print(f'Age std: {age.std():.2f} years')

Age range: 79.58 years (0.42 to 80.0)
Age variance: 211.02 (years squared)
Age std: 14.53 years


### `ddof`: Why Pandas and NumPy Disagree
Compute the standard deviation of the same column two ways:

In [6]:
known_ages = age.dropna()

print('pandas .std():', known_ages.std())
print('np.std():     ', np.std(known_ages))

pandas .std(): 14.526497332334042
np.std():      14.516321150817316


They're different, and neither is a bug. The difference is the **`ddof`** ("delta degrees of freedom") setting:

| | Default | Divides by | Use when |
|---|---|---|---|
| pandas `.std()` / `.var()` | `ddof=1` | n − 1 | your data are a **sample** of a bigger population |
| NumPy `np.std()` / `np.var()` | `ddof=0` | n | your data **are** the whole population |

Dividing by n − 1 corrects for the fact that a sample tends to look a little less spread out than the population it came from. Titanic's 891 rows are a sample of the roughly 2,200 people aboard, so pandas' default is the right one here. With large n the difference is tiny. With 5 values, it isn't.

You can always set it explicitly, and the two will match:

In [7]:
print(known_ages.std(ddof=0))
print(np.std(known_ages, ddof=0))

14.516321150817316
14.516321150817316


## 4. Percentiles, Quartiles, and the IQR
A **percentile** answers "what value is this percentage of the data below?" The 90th percentile of fare is the price that 90% of passengers paid less than.

The **quartiles** are the 25th, 50th, and 75th percentiles (Q1, the median, and Q3). The **interquartile range (IQR)** is Q3 − Q1: the width of the **middle half** of the data. Like the median, it ignores the extremes, so it's robust.

In [8]:
fare.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

0.10      7.55000
0.25      7.91040
0.50     14.45420
0.75     31.00000
0.90     77.95830
0.99    249.00622
Name: Fare, dtype: float64

In [9]:
q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1

print(f'Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Fare std = {fare.std():.2f}')

Q1 = 7.91, Q3 = 31.00, IQR = 23.09
Fare std = 49.69


The middle half of passengers paid within a band about 23 wide, but the standard deviation is about 50. The std is inflated by the same expensive tickets that pulled the mean up. **Robust center (median) goes with robust spread (IQR); mean goes with std.**

Notice the jump from the 90th percentile (about 78) to the 99th (about 250). The top 1% of fares are in a different world from everyone else. Notebook 03 takes this apart.

## 5. Reading `.describe()` Row by Row
You've run `.describe()` since Module 06. Now you know what every row means:

In [10]:
titanic[['Age', 'Fare', 'SibSp']].describe()

,Age,Fare,SibSp
count,714.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008
std,14.526497,49.693429,1.102743
min,0.420000,0.000000,0.000000
25%,20.125000,7.910400,0.000000
50%,28.000000,14.454200,0.000000
75%,38.000000,31.000000,1.000000
max,80.000000,512.329200,8.000000


| Row | Meaning | What to look for |
|---|---|---|
| `count` | Non-missing values | Lower than the number of rows? Missing data. `Age` has 714 of 891. |
| `mean` / `50%` | Mean and median | Far apart? Lopsided data or outliers (`Fare`). |
| `std` | Standard deviation | Large relative to the mean? Very spread out. |
| `min` / `max` | The extremes | Impossible values? (A negative age, a $0 fare.) |
| `25%` / `75%` | Q1 and Q3 | Their difference is the IQR. |

You can ask for different percentiles, and turn the table sideways (`.T`) when there are many columns:

In [11]:
titanic[['Age', 'Fare']].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2)

,count,mean,std,min,5%,50%,95%,max
Age,714.0,29.7,14.53,0.42,4.00,28.00,56.00,80.00
Fare,891.0,32.2,49.69,0.00,7.22,14.45,112.08,512.33


### 🧠 Quick Check
`Fare` has a `min` of `0.00`. Is a $0 fare a data error, a placeholder (Module 06), or something real? How could you investigate? (Hint: filter for those rows and look at the other columns.)

In [12]:
titanic[titanic['Fare'] == 0]

# The $0 fares are not placeholders; they are missing values in the original data.

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
179,180,0,3,"Leonard, Mr. Lionel",male,36.0,0,0,LINE,0.0,NaN,S
263,264,0,1,"Harrison, Mr. William",male,40.0,0,0,112059,0.0,B94,S
271,272,1,3,"Tornquist, Mr. William Henry",male,25.0,0,0,LINE,0.0,NaN,S
277,278,0,2,"Parkes, Mr. Francis ""Frank""",male,NaN,0,0,239853,0.0,NaN,S
302,303,0,3,"Johnson, Mr. William Cahoone Jr",male,19.0,0,0,LINE,0.0,NaN,S
413,414,0,2,"Cunningham, Mr. Alfred Fleming",male,NaN,0,0,239853,0.0,NaN,S
466,467,0,2,"Campbell, Mr. William",male,NaN,0,0,239853,0.0,NaN,S
481,482,0,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0,NaN,S
597,598,0,3,"Johnson, Mr. Alfred",male,49.0,0,0,LINE,0.0,NaN,S
633,634,0,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.0,NaN,S


## 6. Summarizing Categorical Columns
`.describe()` on text columns gives a different summary: count, number of distinct values, the most common value (`top`), and how often it appears (`freq`).

In [13]:
titanic[['Sex', 'Embarked']].describe()

,Sex,Embarked
count,891,889
unique,2,3
top,male,S
freq,577,644


In [14]:
# The proportion in each category: the categorical version of a "summary"
titanic['Embarked'].value_counts(normalize=True, dropna=False).round(3)

Embarked
S      0.723
C      0.189
Q      0.086
NaN    0.002
Name: proportion, dtype: float64

## 7. Missing Values Change the Statistics
Pandas statistics **skip** missing values by default (`skipna=True`). So `age.mean()` is the mean of the **714 known ages**, not all 891 passengers.

In Module 06 you learned to fill missing values. Watch what a "harmless" median fill does to the spread:

In [15]:
age_filled = age.fillna(age.median())

comparison = pd.DataFrame({
    'original': age.describe(),
    'median-filled': age_filled.describe(),
}).round(2)
comparison

,original,median-filled
count,714.00,891.00
mean,29.70,29.36
std,14.53,13.02
min,0.42,0.42
25%,20.12,22.00
50%,28.00,28.00
75%,38.00,35.00
max,80.00,80.00


The median didn't move, but the **std shrank** and the quartiles squeezed toward the middle. You added 177 identical values right at the center, so the data look *less* spread out than they really are.

> ⚠️ Filling missing values is a decision with consequences. If you fill, say so, and compare the statistics before and after, the way the Module 06 lab compared raw and cleaned data.

---
### 📝 Summary
- Center: `.mean()`, `.median()`, `.mode()`. The median is robust to outliers, and the mean is not.
- Spread: range, `.var()`, `.std()`, `.quantile()`, and IQR = Q3 − Q1. Pair median with IQR, and mean with std.
- Pandas uses `ddof=1` (sample) and NumPy uses `ddof=0` (population). Set it explicitly when it matters.
- `.describe()`: `count` reveals missing data, and a big mean-vs-median gap reveals lopsided data.
- Statistics skip `NaN` by default, and filling missing values can quietly shrink the spread.

## 🏋️ Practice

### Practice 1 — Center of Pokémon Stats (easy)
Load `pokemon.csv` into `poke`. For `HP`, `Attack`, and `Speed`, print the mean and median. Which column has the biggest gap between them?

In [16]:
poke = pd.read_csv('pokemon.csv')
# Your code here
print("HP mean:", poke['HP'].mean())
print("HP median:", poke['HP'].median())

print("Attack mean:", poke['Attack'].mean())
print("Attack median:", poke['Attack'].median())

print("Speed mean:", poke['Speed'].mean())
print("Speed median:", poke['Speed'].median())

# HP has the biggest gap between its mean and median.

HP mean: 69.25875
HP median: 65.0
Attack mean: 79.00125
Attack median: 75.0
Speed mean: 68.2775
Speed median: 65.0


### Practice 2 — Same Mean, Different Spread (medium)
Create two Series with **the same mean** (for example, both average to 50) but very different standard deviations. Print the mean, std, and IQR of each. In a comment, describe how a group of people with each set of scores would be different.

In [17]:
# Your code here
spread_scores = pd.Series([0, 10, 20, 30, 40, 60, 70, 80, 90, 100])
close_scores = pd.Series([49, 49, 49, 49, 49, 51, 51, 51, 51, 51])

print("Spread scores mean:", spread_scores.mean())
print("Spread scores std:", spread_scores.std())
print("Spread scores IQR:", spread_scores.quantile(0.75) - spread_scores.quantile(0.25))

print("Close scores mean:", close_scores.mean())
print("Close scores std:", close_scores.std())
print("Close scores IQR:", close_scores.quantile(0.75) - close_scores.quantile(0.25))

# The first group has scores spread across the full range, while the second group has scores clustered tightly around 50.

Spread scores mean: 50.0
Spread scores std: 34.96029493900505
Spread scores IQR: 55.0
Close scores mean: 50.0
Close scores std: 1.0540925533894598
Close scores IQR: 2.0


### Practice 3 — Zero Fares (harder)
Filter `titanic` for rows where `Fare == 0`. How many are there? Look at their `Pclass`, `Age`, `Ticket`, and `Embarked` columns. Then compute the mean and median `Fare` with and without them. In a Markdown cell, say whether you'd treat them as placeholders, and why.

In [18]:
# Your code here
zero_fares = titanic[titanic['Fare'] == 0]
print("Rows with Fare == 0:", len(zero_fares))
print(zero_fares[['Pclass', 'Age', 'Ticket', 'Embarked']])

print("Fare mean with zeros:", titanic['Fare'].mean())
print("Fare median with zeros:", titanic['Fare'].median())
print("Fare mean without zeros:", titanic.loc[titanic['Fare'] != 0, 'Fare'].mean())
print("Fare median without zeros:", titanic.loc[titanic['Fare'] != 0, 'Fare'].median())

Rows with Fare == 0: 15
     Pclass   Age  Ticket Embarked
179       3  36.0    LINE        S
263       1  40.0  112059        S
271       3  25.0    LINE        S
277       2   NaN  239853        S
302       3  19.0    LINE        S
413       2   NaN  239853        S
466       2   NaN  239853        S
481       2   NaN  239854        S
597       3  49.0    LINE        S
633       1   NaN  112052        S
674       2   NaN  239856        S
732       2   NaN  239855        S
806       1  39.0  112050        S
815       1   NaN  112058        S
822       1  38.0   19972        S
Fare mean with zeros: 32.204207968574636
Fare median with zeros: 14.4542
Fare mean without zeros: 32.75564988584475
Fare median without zeros: 14.5


*I would treat them as placeholders. The zero fares are concentrated in third class, and many have missing ages and embarked values, suggesting that the fare was unknown rather than genuinely zero.*

## ✏️ Your Turn
Pick one numeric column from `pokemon.csv`. In 2–3 sentences, describe it using **one center and one spread statistic that fit its shape**, and justify your choice using the gap between its mean and median.

In [19]:
# Your turn here
poke['HP'].describe()

count    800.000000
mean      69.258750
std       25.534669
min        1.000000
25%       50.000000
50%       65.000000
75%       80.000000
max      255.000000
Name: HP, dtype: float64

The mean and median are close, so the distribution is roughly symmetric. Therefore, I would describe its center with the mean and its spread with the standard deviation.

## 🔥 Challenge (Optional) — Median Absolute Deviation
The **median absolute deviation (MAD)** is a robust replacement for the standard deviation: the median of the distances between each value and the median.
```python
mad = (s - s.median()).abs().median()
```
Compute the MAD and the std for `fares` and `with_outlier` from section 2. How much does each one change when the outlier is added?

In [20]:
# Your code here

---
✅ Next up → [Shape and Outliers](03-Shape_and_Outliers.ipynb): measuring how lopsided a column is, and finding the values that don't fit.